# 08 · Missing Ablations & Runtime (paper gaps)

**Purpose.** Close the three gaps the paper outline flags: (1) **without-alias-map** ablation, (2) **direct-only vs tblastn-only vs hybrid** routing, (3) **runtime** benchmark.

## Inputs

All three virus datasets.

In [ ]:
from pathlib import Path
import sys, time

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA   = ROOT / "app" / "data"
CONFIG = ROOT / "app" / "config"

# References (verify these are the intended ref records for the paper):
FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"        # alt: FMD_FJ175661_Anno.gb
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"       # alt: PRRS_MT746146_Anno.gb
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REFS   = {"ref_1": DATA / "PED" / "PED_ref_1.gb",
              "ref_2": DATA / "PED" / "PED_ref_2.gb"}
PED_QUERY  = DATA / "PED" / "PED_100seqs.gb"

# Run toggle: keep False for a fast smoke test, True for the full 100-record run.
RUN_FULL = False
SAMPLE_N = 10


In [ ]:
# outputs land inside this unit folder so figures/tables sit next to the notebook
UNIT_DIR = ROOT / "app" / "validation" / "07_ablation_runtime"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## (1) Without alias map

> ⚠️ **TODO**: run the pipeline with an empty/identity alias_lookup and re-score to show how much name-standardisation contributes. Pass an empty lookup into the compare step.

## (2) Routing ablation: direct-only vs tblastn-only vs hybrid

> ⚠️ **TODO**: force `strategy='direct'` for all records, then `strategy='tblastn'` for all, then the default hybrid router; compare accuracy across the three.

In [ ]:
from app.validation._shared.validation_utils import (
    run_tblastn_against_truth, run_production_pipeline_against_truth, summarize_comparison,
)

pairs = {"fmd": (FMD_REF, FMD_QUERY), "prrs": (PRRS_REF, PRRS_QUERY), "ped": (PED_REFS["ref_1"], PED_QUERY)}

# hybrid (default router)
hybrid = []
for label, (ref, query) in pairs.items():
    pp, _ = run_production_pipeline_against_truth(label, ref, query, OUT / "hybrid" / label)
    pp["virus"] = label; pp["mode"] = "hybrid"; hybrid.append(pp)

# tblastn-only
tbl = []
for label, (ref, query) in pairs.items():
    pp, _ = run_tblastn_against_truth(label, ref, query, OUT / "tblastn_only" / label)
    pp["virus"] = label; pp["mode"] = "tblastn_only"; tbl.append(pp)

# direct-only -- TODO: force direct_extract_with_alias for all records and score
routing = pd.concat(hybrid + tbl, ignore_index=True)
summarize_comparison(routing, ["mode", "virus"])

## (3) Runtime benchmark

In [ ]:
import time
timings = []
for label, (ref, query) in pairs.items():
    t0 = time.perf_counter()
    run_tblastn_against_truth(label, ref, query, OUT / "timing" / label, progress=False)
    timings.append({"virus": label, "seconds": round(time.perf_counter() - t0, 2)})
timing_df = pd.DataFrame(timings)
timing_df.to_csv(OUT / "runtime.tsv", sep="\t", index=False)
timing_df

## Interpretation

> ⚠️ **TODO**: alias map và hybrid routing đều đóng góp accuracy; runtime chấp nhận được cho 100 records.